In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from pyspark.sql.functions import current_timestamp, to_timestamp, concat, col, lit

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsproysmartdata01")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/Catalogo_ClienteProducto.csv"


In [0]:
CatalogoCliente_schema = StructType(fields=[
    StructField("id_cliente", IntegerType(), False),
    StructField("correlativo", IntegerType(), False),
    StructField("cliente", StringType(), True),
    StructField("PRODUCTO", StringType(), True)
])

In [0]:
df_catalogo = spark.read\
.option('header', True)\
.schema(CatalogoCliente_schema)\
.csv(ruta)

In [0]:
catalogo_final_df = df_catalogo.select(
    col("id_cliente"),
    col("correlativo").alias("id_producto"),
    col("cliente"),
    col("PRODUCTO").alias("producto")
).withColumn("ingestion_date", current_timestamp())

In [0]:
catalogo_final_df.write \
    .mode("overwrite") \
    .insertInto(f"{catalogo}.{esquema}.catalogo_cliente_producto")

In [0]:
%sql
SELECT * FROM catalog_au.bronze.catalogo_cliente_producto